Pipeline taxonomy
-----------------
The 28 distinct values of apprehension_method are consolidated into five
analytical categories based on the administrative system ICE queried:

  CAP (Criminal Alien Program)
      Includes: any method containing "CAP"
      Logic: Screens jail, prison, and criminal database records. Finds
             anyone with any prior criminal justice contact.

  Community Enforcement
      Includes: "Non-Custodial Arrest" and variants
      Logic: Street-level enforcement. No prior database contact required.

  287(g) Police Database
      Includes: any method containing "287"
      Logic: Local law enforcement databases, deputized by ICE.

  Fugitive / Located
      Includes: methods containing "Fugitive" or "Located"
      Logic: Tip-based or prior-order enforcement.

  Border / Patrol
      Includes: methods containing "Patrol", "Boat", or "Inspection"
      Logic: Border crossing or maritime enforcement.

Outcome variable
----------------
deported = 1 if case_status contains "Deported", "Removed", or "Excluded"
deported = 0 otherwise

Input
-----
data/raw/arrests_core.csv
Columns used: apprehension_method, apprehension_criminality, case_criminality,
              case_threat_level, final_program, case_status, citizenship_country,
              gender, apprehension_state, apprehension_date, birth_year

Output
------
data/interim/arrests_classified.csv


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import pandas as pd
from config import DATA_RAW, DATA_INTERIM


PIPELINE_MAP = {
    "CAP":       ("CAP",),
    "287g":      ("287",),
    "Community": ("Non-Custodial",),
    "Worksite":  ("Worksite",),
    "Border":    ("Patrol", "Boat", "Inspection"),
    "Fugitive":  ("Fugitive", "Located"),
}


def classify_pipeline(method: str) -> str:
    m = str(method).strip()
    for label, keywords in PIPELINE_MAP.items():
        if any(kw in m for kw in keywords):
            return label
    return "Other"


def main():
    path = DATA_RAW / "arrests_core.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\n"
            "Export from Colab using the preprocessing notebook:\n"
            "  notebooks/00_preprocess_raw_ddp.ipynb\n"
            "Required columns: apprehension_method, apprehension_criminality,\n"
            "  case_criminality, case_threat_level, final_program, case_status,\n"
            "  citizenship_country, gender, apprehension_state, apprehension_date"
        )

    print(f"Loading arrests from {path.name}...")
    df = pd.read_csv(path, low_memory=False)
    print(f"  Loaded {len(df):,} rows, {len(df.columns)} columns")

    # ── Pipeline classification ────────────────────────────────────────────────
    df["pipeline"] = df["apprehension_method"].apply(classify_pipeline)

    print("\nPipeline distribution:")
    print(df["pipeline"].value_counts().to_string())

    # ── Outcome: deportation ──────────────────────────────────────────────────
    df["deported"] = (
        df["case_status"]
        .fillna("")
        .str.contains("Deported|Removed|Excluded", case=False, na=False)
        .astype(int)
    )
    print(f"\nOverall deportation rate: {df['deported'].mean():.1%}")

    # ── Year extraction ───────────────────────────────────────────────────────
    df["year"] = pd.to_datetime(df["apprehension_date"], errors="coerce").dt.year

    # ── Save ──────────────────────────────────────────────────────────────────
    out = DATA_INTERIM / "arrests_classified.csv"
    df.to_csv(out, index=False)
    print(f"\nSaved → {out}  ({len(df):,} rows)")


if __name__ == "__main__":
    main()
